In [ ]:
import pandas as pd #Librería para analizar y manipular datos, en formato de tablas 
import os #Para interactuar con el sistema operativo (rutas, carpetas, archivos)
import glob #Para buscar archivos específicos dentro de una carpeta


#Configuración de rutas
#Defino una variable, con la ruta al cual quiero acceder
PATH_DATA = "colocar la ruta"   
                                                   

#Creo una función 
def cargar_y_analizar():
    
    #Buscamos todos los archivos csv y excel, usando comodines "*.csv"
    #glob es una módulo y .glob es una función
    #os.path.join arma la ruta / os es un módulo de python / patch un submodulo / join es una función
    #+ \ esto sirve para concatenar listas
        
    archivos = glob.glob(os.path.join(PATH_DATA, "*.csv")) + \
               glob.glob(os.path.join(PATH_DATA, "*.xlsx"))
    
    #Creo un diccionario vacio, para guardar los DataFrames por separado
    tablas = {}

    print(f"Se encontraron {len(archivos)} archivos.") #usamos la función len, porque este suma elementos

    for ruta in archivos: #variable temporal que apunta a cada elemento de la lista (archivos) mientras recorre el bucle
        nombre_archivo = os.path.basename(ruta) #extrae el nombre del archivo
        nombre_tabla = os.path.splitext(nombre_archivo)[0] #quita la extensión
        
        #Cargas según su extensión
        try: #esto ejecuta este bloque de código, y si ocurre un error, no se detendra  el programa
            if ruta.endswith('.csv'): #la función endswith, verifica si el archivo termina con .csv o excel
                df = pd.read_csv(ruta)
            else:
                df = pd.read_excel(ruta)
            
            #Guardamos en el diccionario
            tablas[nombre_tabla] = df
            
            #ETL básico por tabla
            print(f"TABLA: {nombre_tabla} ")
            print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
            
            #Ver tipos de datos y nulos
            info_df = pd.DataFrame({
                'Columna': df.columns,
                'Tipo': df.dtypes.values,
                'Nulos': df.isnull().sum().values,
                '% Nulos': (df.isnull().sum().values / len(df) * 100).round(2)
            })
            print(info_df)
            
            #Mostrar las primeras 3 filas de la tabla
            print("\nPrimeras filas:")
            print(df.head(3))
            print("*" * 40 + "\n")
            
        except Exception as e:
            print(f"Error cargando {nombre_archivo}: {e}")

    return tablas

#Ejecutar el proceso
mis_tablas = cargar_y_analizar()

Se encontraron 5 archivos.
TABLA: calendar 
Dimensiones: 731 filas x 6 columnas
       Columna    Tipo  Nulos  % Nulos
0         date  object      0      0.0
1         year   int64      0      0.0
2        month   int64      0      0.0
3          day   int64      0      0.0
4         week   int64      0      0.0
5  day_of_week   int64      0      0.0

Primeras filas:
         date  year  month  day  week  day_of_week
0  2023-01-01  2023      1    1    52            6
1  2023-01-02  2023      1    2     1            0
2  2023-01-03  2023      1    3     1            1
****************************************

TABLA: customers 
Dimensiones: 50000 filas x 5 columnas
          Columna    Tipo  Nulos  % Nulos
0     customer_id  object      0      0.0
1             age   int64      0      0.0
2          gender  object      0      0.0
3  loyalty_member   int64      0      0.0
4       join_date  object      0      0.0

Primeras filas:
  customer_id  age  gender  loyalty_member   join_date
0   

In [46]:
#Primero extraigo las tablas del diccionario 'mis_tablas'
df_sales = mis_tablas.get('sales')     
df_products = mis_tablas.get('products') 

#Ahora sí, corro el merge
audit = pd.merge(df_sales, df_products, on='product_id', how='left', indicator=True)

#Filtramos los huérfanos
faltantes = audit[audit['_merge'] == 'left_only']

#Sacamos los IDs únicos
ids_a_crear = faltantes['product_id'].unique()

print(f"Productos detectados en Ventas que NO FUERON ALTEADOS: {ids_a_crear}")

Productos detectados en Ventas que NO FUERON ALTEADOS: ['P0000' 'P0201']


In [47]:
#Extraemos la tabla original
df_products = mis_tablas.get('products')

#Definimos los datos (CUIDADO: Todos deben tener 2 elementos)
data_nuevos = {
    'product_id': ['P0000', 'P0201'], 
    'product_name': ['Chocolate Recuperado A', 'Chocolate Recuperado B'],
    'brand': ['Desconocida', 'Desconocida'],
    'category': ['Sin Categoría', 'Sin Categoría'],
    'cocoa_percent': [0, 0],   # <--- Antes quizás tenías solo uno
    'weight_g': [0, 0]         # <--- Antes quizás tenías solo uno
}

#Ahora sí, creamos el DataFrame sin errores
df_nuevos = pd.DataFrame(data_nuevos)

#Concatenamos
df_products_actualizado = pd.concat([df_products, df_nuevos], ignore_index=True)

#Verificación final
print(f"Ahora la tabla tiene {len(df_products_actualizado)} filas.")
print(df_products_actualizado.tail(2))

# Guardamos en el diccionario para seguir el proceso
mis_tablas['products'] = df_products_actualizado

Ahora la tabla tiene 202 filas.
    product_id            product_name        brand       category  \
200      P0000  Chocolate Recuperado A  Desconocida  Sin Categoría   
201      P0201  Chocolate Recuperado B  Desconocida  Sin Categoría   

     cocoa_percent  weight_g  
200              0         0  
201              0         0  


In [58]:
#Trabajamos sobre el DataFrame actualizado que ya tiene los chocolates nuevos
df_products = mis_tablas['products'].copy()

# unción con condición: Si ya es número, no lo toca. Si tiene letras, las quita.
def limpiar_id_producto(id_val):
    if pd.isna(id_val): return 0
    id_str = str(id_val).upper().strip()
    if 'P' in id_str:
        #Extrae solo los números. 'P0000' -> '0000' -> 0
        solo_numeros = re.sub(r'\D', '', id_str)
        return int(solo_numeros) if solo_numeros != '' else 0
    try:
        return int(float(id_str))
    except:
        return 0

df_products['product_id'] = df_products['product_id'].apply(limpiar_id_producto)
mis_tablas['products'] = df_products
print("PRODUCTS CORREGIDO")
print(df_products) # Ajusta 'customer_id' si se llama distinto
print(f"\nTipo de dato confirmado: {df_products['product_id'].dtype}")

PRODUCTS CORREGIDO
     product_id            product_name        brand       category  \
0             1     White Chocolate 80%         Mars        Truffle   
1             2      Dark Chocolate 70%      Cadbury        Praline   
2             3   Truffle Chocolate 70%      Hershey        Praline   
3             4      Milk Chocolate 50%         Mars        Praline   
4             5     White Chocolate 70%      Ferrero          White   
..          ...                     ...          ...            ...   
197         198      Dark Chocolate 90%      Ferrero          White   
198         199      Dark Chocolate 70%      Hershey          White   
199         200      Milk Chocolate 50%      Cadbury           Milk   
200           0  Chocolate Recuperado A  Desconocida  Sin Categoría   
201         201  Chocolate Recuperado B  Desconocida  Sin Categoría   

     cocoa_percent  weight_g  
0               80       120  
1               70       100  
2               70       120  
3   

In [56]:
import re

#Cargamos de nuevo el original del diccionario para pisar los ceros
df_customers = mis_tablas['customers'].copy()

def limpiar_id_universal(id_val):
    if pd.isna(id_val): return 0
    id_str = str(id_val).strip()
    # re.sub(r'\D', '', ...) elimina letras, espacios, guiones, TODO.
    solo_numeros = re.sub(r'\D', '', id_str)
    
    try:
        #Si quedó algo, lo hace entero. 'C0001' -> '0001' -> 1
        return int(solo_numeros) if solo_numeros != '' else 0
    except:
        return 0

#Aplicamos la limpieza universal
df_customers['customer_id'] = df_customers['customer_id'].apply(limpiar_id_universal)

#IMPORTANTE: Aseguramos que sea tipo int
df_customers['customer_id'] = df_customers['customer_id'].astype(int)

#Guardamos el cambio real
mis_tablas['customers'] = df_customers

print("CUSTOMERS CORREGIDO")
print(df_customers) # Ajusta 'customer_id' si se llama distinto
print(f"\nTipo de dato confirmado: {df_customers['customer_id'].dtype}")

CUSTOMERS CORREGIDO
       customer_id  age  gender  loyalty_member   join_date
0                1   40    Male               1  2025-05-21
1                2   47    Male               0  2021-12-26
2                3   58  Female               1  2022-09-13
3                4   25  Female               0  2025-02-27
4                5   43    Male               0  2023-08-31
...            ...  ...     ...             ...         ...
49995        49996   30  Female               1  2022-11-21
49996        49997   24  Female               0  2022-08-30
49997        49998   53    Male               0  2024-09-22
49998        49999   31  Female               1  2025-08-23
49999        50000   39  Female               1  2023-12-02

[50000 rows x 5 columns]

Tipo de dato confirmado: int64


In [57]:
import re

#Extraemos la copia de la tabla original
df_stores = mis_tablas['stores'].copy()

#Definimos la función universal (la "Barre-Letras")
def limpiar_id_universal(id_val):
    if pd.isna(id_val): return 0
    id_str = str(id_val).strip()
    
    # \D elimina cualquier cosa que NO sea un número
    solo_numeros = re.sub(r'\D', '', id_str)
    
    try:
        # Convertimos a entero. Ejemplo: 'S0001' -> '0001' -> 1
        return int(solo_numeros) if solo_numeros != '' else 0
    except:
        return 0

#Aplicamos a la columna de ID
df_stores['store_id'] = df_stores['store_id'].apply(limpiar_id_universal)

#Aseguramos el tipo de dato numérico
df_stores['store_id'] = df_stores['store_id'].astype(int)

#Guardamos en el diccionario actualizado
mis_tablas['stores'] = df_stores

print("STORES LIMPIO")
print(df_stores) # Ajusta 'store_name' si se llama distinto
print(f"\nTipo de dato confirmado: {df_stores['store_id'].dtype}")

STORES LIMPIO
    store_id           store_name       city    country store_type
0          1    Chocolate Store 1   New York     Canada     Retail
1          2    Chocolate Store 2  Melbourne     Canada       Mall
2          3    Chocolate Store 3     Berlin     France       Mall
3          4    Chocolate Store 4      Paris         UK    Airport
4          5    Chocolate Store 5     Sydney        USA     Online
..       ...                  ...        ...        ...        ...
95        96   Chocolate Store 96     Berlin         UK       Mall
96        97   Chocolate Store 97    Toronto     Canada       Mall
97        98   Chocolate Store 98   New York  Australia     Online
98        99   Chocolate Store 99    Toronto  Australia       Mall
99       100  Chocolate Store 100     Berlin     France    Airport

[100 rows x 5 columns]

Tipo de dato confirmado: int64


In [60]:
import re

#Obtenemos la tabla de ventas
df_sales = mis_tablas['sales'].copy()

#Definimos la función universal (la misma que usamos en los maestros)
def limpiar_id_universal(id_val):
    if pd.isna(id_val): return 0
    id_str = str(id_val).strip()
    
    # \D elimina todo lo que no sea un número (letras, guiones, espacios)
    solo_numeros = re.sub(r'\D', '', id_str)
    
    try:
        # Convertimos a entero. Ej: 'P0201' -> '201', 'C0005' -> '5'
        return int(solo_numeros) if solo_numeros != '' else 0
    except:
        return 0

#Aplicamos la limpieza a los 3 IDs que mencionaste (y al Order_ID de paso)
print("Limpiando IDs en Sales...")
df_sales['product_id'] = df_sales['product_id'].apply(limpiar_id_universal)
df_sales['customer_id'] = df_sales['customer_id'].apply(limpiar_id_universal)
df_sales['store_id'] = df_sales['store_id'].apply(limpiar_id_universal)

#Si tu tabla de ventas tiene order_id, también límpialo
if 'order_id' in df_sales.columns:
    df_sales['order_id'] = df_sales['order_id'].apply(limpiar_id_universal)

#Forzamos que todos sean tipo entero (INT)
cols_id = ['product_id', 'customer_id', 'store_id']
if 'order_id' in df_sales.columns: cols_id.append('order_id')

df_sales[cols_id] = df_sales[cols_id].astype(int)

#Guardamos la tabla definitiva en el diccionario
mis_tablas['sales'] = df_sales

print("Sincronización completada con éxito.")

#VERIFICACIÓN FINAL
print("\nPrimeras filas de Sales con IDs limpios:")
print(df_sales[cols_id].head())
print(f"\nTipos de datos finales en Sales:\n{df_sales[cols_id].dtypes}")

Limpiando IDs en Sales...
Sincronización completada con éxito.

Primeras filas de Sales con IDs limpios:
   product_id  customer_id  store_id  order_id
0          80        40749        93         1
1         173        20161        65         2
2         115        48069        78         3
3         186        47901        88         4
4         197        33950        54         5

Tipos de datos finales en Sales:
product_id     int64
customer_id    int64
store_id       int64
order_id       int64
dtype: object


In [72]:
#Renombrar 'date' a 'CALENDAR_DATE' para evitar conflictos con la palabra reservada de Oracle
if 'date' in mis_tablas['calendar'].columns:
    mis_tablas['calendar'] = mis_tablas['calendar'].rename(columns={'date': 'calendar_date'})

#Convertir todas las columnas de fecha de 'object' a 'datetime' real
tablas_con_fecha = {
    'calendar': 'calendar_date',
    'sales': 'order_date',
    'customers': 'join_date'
}

for tabla, col_fecha in tablas_con_fecha.items():
    if tabla in mis_tablas:
        mis_tablas[tabla][col_fecha] = pd.to_datetime(mis_tablas[tabla][col_fecha])
        print(f"Fecha en {tabla}.{col_fecha} convertida a datetime64")

#Verificación rápida
print("\nConfirmación de tipos de fecha:")
print(f"Calendar: {mis_tablas['calendar']['calendar_date'].dtype}")
print(f"Sales: {mis_tablas['sales']['order_date'].dtype}")

Fecha en calendar.calendar_date convertida a datetime64
Fecha en sales.order_date convertida a datetime64
Fecha en customers.join_date convertida a datetime64

Confirmación de tipos de fecha:
Calendar: datetime64[ns]
Sales: datetime64[ns]


In [73]:
#Extraemos TODAS las tablas necesarias
df_sales = mis_tablas.get('sales')
df_products = mis_tablas.get('products')  
df_customers = mis_tablas.get('customers')
df_stores = mis_tablas.get('stores')       

def check_integrity(df_hechos, df_maestro, fk, nombre_maestro):
    if df_maestro is None:
        print(f"ADVERTENCIA: No se encontró la tabla {nombre_maestro} en el diccionario.")
        return

    #Merge de auditoría
    audit = pd.merge(df_hechos, df_maestro, on=fk, how='left', indicator=True)
    
    #Identificar huérfanos
    missing = audit[audit['_merge'] == 'left_only'][fk].unique()
    
    if len(missing) > 0:
        print(f"ERROR en {nombre_maestro}: Se encontraron {len(missing)} IDs huérfanos.")
        print(f"IDs a revisar en Sales: {missing}")
    else:
        print(f"{nombre_maestro}: Integridad perfecta. No faltan IDs.")

# 2. Corremos el escáner total
print("AUDITORÍA INTEGRAL DE RELACIONES")
check_integrity(df_sales, df_products, 'product_id', 'PRODUCTOS')
check_integrity(df_sales, df_customers, 'customer_id', 'CLIENTES')
check_integrity(df_sales, df_stores, 'store_id', 'TIENDAS') #Validación de tiendas

AUDITORÍA INTEGRAL DE RELACIONES
PRODUCTOS: Integridad perfecta. No faltan IDs.
CLIENTES: Integridad perfecta. No faltan IDs.
TIENDAS: Integridad perfecta. No faltan IDs.


In [75]:
print("ESTADO ACTUAL DE LOS DATOS (dtypes)\n")
print("="*50)

for nombre_tabla, df in mis_tablas.items():
    print(f"\nTABLA: {nombre_tabla.upper()}")
    print("-" * 20)
    
    #Mostramos los dtypes
    print(df.dtypes)
    
    #Pequeña auditoría visual rápida
    cols_id = [c for c in df.columns if 'id' in c.lower()]
    cols_date = [c for c in df.columns if 'date' in c.lower()]
    
    if cols_id:
        print(f"👉 IDs detectados: {cols_id} (Deberían ser int64)")
    if cols_date:
        print(f"👉 Fechas detectadas: {cols_date} (Deberían ser datetime64 o object)")
    
    print(f"Total registros: {len(df)}")
    print("-" * 50)

ESTADO ACTUAL DE LOS DATOS (dtypes)


TABLA: CALENDAR
--------------------
calendar_date    datetime64[ns]
year                      int64
month                     int64
day                       int64
week                      int64
day_of_week               int64
dtype: object
👉 Fechas detectadas: ['calendar_date'] (Deberían ser datetime64 o object)
Total registros: 731
--------------------------------------------------

TABLA: CUSTOMERS
--------------------
customer_id                int64
age                        int64
gender                    object
loyalty_member             int64
join_date         datetime64[ns]
dtype: object
👉 IDs detectados: ['customer_id'] (Deberían ser int64)
👉 Fechas detectadas: ['join_date'] (Deberían ser datetime64 o object)
Total registros: 50000
--------------------------------------------------

TABLA: PRODUCTS
--------------------
product_id        int64
product_name     object
brand            object
category         object
cocoa_percent     int6

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import urllib.parse

#CONFIGURACIÓN DE CONEXIÓN
usuario, password, host, puerto, servicio = 'xxxxxxxxxx', 'xxxxxxxxxx', 'xxxxxxxxxx', 'xxxx', 'xx'
password_escrita = urllib.parse.quote_plus(password)
url = f'oracle+oracledb://{usuario}:{password_escrita}@{host}:{puerto}/?service_name={servicio}'
engine = create_engine(url)

#DEFINICIÓN DE MOLDES SQL
moldes = {
    "calendar": """
        CREATE TABLE calendar (
            CALENDAR_DATE DATE PRIMARY KEY,
            YEAR NUMBER,
            MONTH NUMBER,
            DAY NUMBER,
            DAY_NAME VARCHAR2(20)
        )""",
    "products": """
        CREATE TABLE products (
            PRODUCT_ID NUMBER PRIMARY KEY,
            PRODUCT_NAME VARCHAR2(50),
            BRAND VARCHAR2(50),
            CATEGORY VARCHAR2(50),
            COCOA_PERCENT NUMBER,
            WEIGHT_G NUMBER
        )""",
    "customers": """
        CREATE TABLE customers (
            CUSTOMER_ID NUMBER PRIMARY KEY,
            AGE NUMBER,
            GENDER VARCHAR2(20),
            LOYALTY_MEMBER NUMBER,
            JOIN_DATE DATE
        )""",
    "stores": """
        CREATE TABLE stores (
            STORE_ID NUMBER PRIMARY KEY,
            STORE_NAME VARCHAR2(50),
            CITY VARCHAR2(50),
            COUNTRY VARCHAR2(50),
            STORE_TYPE VARCHAR2(50)
        )""",
    "sales": """
        CREATE TABLE sales (
            ORDER_ID NUMBER PRIMARY KEY,
            ORDER_DATE DATE,
            PRODUCT_ID NUMBER,
            STORE_ID NUMBER,
            CUSTOMER_ID NUMBER,
            QUANTITY NUMBER,
            UNIT_PRICE NUMBER,
            DISCOUNT NUMBER,
            REVENUE NUMBER,
            COST NUMBER,
            PROFIT NUMBER
        )"""
}

print("Iniciando Ingesta Final con soporte para tipo DATE")

with engine.connect() as conn:
    
    #PASO 1: CALENDARIO (Con conversión a datetime)
    try:
        print("\nGenerando Calendario Dinámico")
        df_ventas_temp = mis_tablas.get('sales')
        
        # Convertimos a datetime para calcular el máximo correctamente
        ultima_fecha = pd.to_datetime(df_ventas_temp['order_date']).max()
        fecha_final_cal = max(ultima_fecha, pd.Timestamp('2026-12-31'))
        
        fechas = pd.date_range(start='2023-01-01', end=fecha_final_cal)
        df_calendar = pd.DataFrame({
            'CALENDAR_DATE': fechas, # Guardamos como objetos datetime, NO como strings
            'YEAR': fechas.year,
            'MONTH': fechas.month,
            'DAY': fechas.day,
            'DAY_NAME': fechas.day_name()
        })

        try:
            conn.execute(text("DROP TABLE calendar CASCADE CONSTRAINTS"))
            conn.commit()
        except: pass
        
        conn.execute(text(moldes["calendar"]))
        conn.commit()

        df_calendar.to_sql('calendar', con=engine, if_exists='append', index=False)
        print(f"TABLA: calendar lista.")

    except Exception as e:
        print(f"Error en Calendario: {e}")

    #PASO 2: RESTO DE TABLAS (Convertir fechas antes de subir)
    tablas_maestras = ['products', 'customers', 'stores', 'sales']

    for nombre_tabla in tablas_maestras:
        try:
            print(f"\nProcesando {nombre_tabla.upper()}")
            df = mis_tablas.get(nombre_tabla)
            if df is None: continue

            df_para_subir = df.copy()

            #CONVERSIÓN CRÍTICA DE FECHAS
            #Si la tabla es sales o customers, convertimos sus columnas de fecha a datetime real
            if nombre_tabla == 'sales' and 'order_date' in df_para_subir.columns:
                df_para_subir['order_date'] = pd.to_datetime(df_para_subir['order_date'])
            
            if nombre_tabla == 'customers' and 'join_date' in df_para_subir.columns:
                df_para_subir['join_date'] = pd.to_datetime(df_para_subir['join_date'])

            #Columnas a MAYÚSCULAS para Oracle
            df_para_subir.columns = [c.upper() for c in df_para_subir.columns]
            
            try:
                conn.execute(text(f"DROP TABLE {nombre_tabla} CASCADE CONSTRAINTS"))
                conn.commit()
            except: pass
            
            conn.execute(text(moldes[nombre_tabla]))
            conn.commit()
            
            df_para_subir.to_sql(
                name=nombre_tabla, 
                con=engine, 
                if_exists='append', 
                index=False, 
                chunksize=10000 
            )
            
            res = conn.execute(text(f"SELECT COUNT(*) FROM {nombre_tabla}"))
            print(f"ÉXITO: {res.scalar()} filas cargadas.")

        except Exception as e:
            print(f"Error crítico en {nombre_tabla}: {e}")

print("\nPROCESO FINALIZADO: Todo cargado con formatos DATE y NUMBER en Oracle.")

Iniciando Ingesta Final con soporte para tipo DATE

Generando Calendario Dinámico
TABLA: calendar lista.

Procesando PRODUCTS
ÉXITO: 202 filas cargadas.

Procesando CUSTOMERS
ÉXITO: 50000 filas cargadas.

Procesando STORES
ÉXITO: 100 filas cargadas.

Procesando SALES
ÉXITO: 1000000 filas cargadas.

PROCESO FINALIZADO: Todo cargado con formatos DATE y NUMBER en Oracle.
